In [1]:
from simple_asr_dataset import SimpleASRDataset

dataset = SimpleASRDataset().sample()





/home/ubuntu/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating validation split: 100%|██████████| 2703/2703 [00:00<00:00, 2980.77 examples/s]


In [4]:
import io
import soundfile as sf

def calculate_audio_duration(audio_bytes):
    """
    Calculate duration of audio from bytes.
    Works with FLAC, WAV, and other formats supported by soundfile.
    
    Args:
        audio_bytes: Raw audio bytes
        
    Returns:
        duration in seconds
    """
    # Create a file-like object from bytes
    audio_io = io.BytesIO(audio_bytes)
    
    # Read audio file
    info = sf.info(audio_io)
    
    return info.duration

# Calculate duration for each audio in the dataset
for idx, item in enumerate(dataset):
    audio_bytes = item['audio']['bytes']
    duration = calculate_audio_duration(audio_bytes)
    
    print(f"Sample {idx + 1}:")
    print(f"  ID: {item['id']}")
    print(f"  Path: {item['audio']['path']}")
    print(f"  Duration: {duration:.2f} seconds ({duration:.3f}s)")
    print(f"  Text: {item['text'][:50]}...")
    print()


Sample 1:
  ID: 2961-960-0003
  Path: 2961-960-0003.flac
  Duration: 17.32 seconds (17.315s)
  Text: THEY WERE ABSORBED IN HIS THEOLOGY AND WERE UNDER ...

Sample 2:
  ID: 8555-284447-0017
  Path: 8555-284447-0017.flac
  Duration: 11.62 seconds (11.620s)
  Text: WHEN THIS HAD BEEN ACCOMPLISHED THE BOOLOOROO LEAN...

Sample 3:
  ID: 2300-131720-0005
  Path: 2300-131720-0005.flac
  Duration: 6.90 seconds (6.900s)
  Text: WHY IF WE ERECT A STATION AT THE FALLS IT IS A GRE...

Sample 4:
  ID: 5142-36377-0022
  Path: 5142-36377-0022.flac
  Duration: 11.24 seconds (11.245s)
  Text: I WISH YOU GOOD NIGHT SHE LAID HER BONY HANDS ON T...

Sample 5:
  ID: 8455-210777-0040
  Path: 8455-210777-0040.flac
  Duration: 6.20 seconds (6.195s)
  Text: THEN SAID SIR FERDINANDO THERE IS NOTHING FOR IT B...

Sample 6:
  ID: 5683-32866-0023
  Path: 5683-32866-0023.flac
  Duration: 2.75 seconds (2.745s)
  Text: ALL THE FURNITURE BELONGED TO OTHER TIMES...

Sample 7:
  ID: 4992-41806-0001
  Path: 4992-41806-00

In [17]:
# Filter audio files with duration between 29s and 30s (inclusive)
filtered_ids = []
filtered_items = []

for item in dataset:
    audio_bytes = item['audio']['bytes']
    duration = calculate_audio_duration(audio_bytes)
    
    # Keep only audio with 29s <= duration <= 30s
    if duration >= 24.5 :
        filtered_ids.append(item['id'])
        filtered_items.append({
            'id': item['id'],
            'duration': duration,
            'path': item['audio']['path'],
            'text': item['text']
        })

print("=" * 70)
print(f"Audio files with duration between 29s and 30s (inclusive)")
print("=" * 70)
print(f"Total found: {len(filtered_ids)}")
print()

if filtered_items:
    print("Filtered audio details:")
    print("-" * 70)
    for idx, item in enumerate(filtered_items, 1):
        print(f"{idx}. ID: {item['id']}")
        print(f"   Duration: {item['duration']:.3f} seconds")
        print(f"   Path: {item['path']}")
        print(f"   Text: {item['text'][:60]}...")
        print()
else:
    print("No audio files found in the 29-30 second range.")
    
print("=" * 70)
print(f"IDs list: {filtered_ids}")


Audio files with duration between 29s and 30s (inclusive)
Total found: 32

Filtered audio details:
----------------------------------------------------------------------
1. ID: 8224-274381-0002
   Duration: 24.540 seconds
   Path: 8224-274381-0002.flac
   Text: WHILE THE FORMER FORETOLD THAT THE SCOTTISH COVENANTERS WERE...

2. ID: 3570-5696-0003
   Duration: 25.115 seconds
   Path: 3570-5696-0003.flac
   Text: A RECONCILIATION BETWEEN THE TWO CONFLICTING REQUIREMENTS IS...

3. ID: 2961-960-0000
   Duration: 27.180 seconds
   Path: 2961-960-0000.flac
   Text: HE PASSES ABRUPTLY FROM PERSONS TO IDEAS AND NUMBERS AND FRO...

4. ID: 5639-40744-0031
   Duration: 28.420 seconds
   Path: 5639-40744-0031.flac
   Text: SO PERSUASIVE WERE HER ENTREATIES AND SO STRONG HER ASSURANC...

5. ID: 1995-1836-0004
   Duration: 33.910 seconds
   Path: 1995-1836-0004.flac
   Text: AS SHE AWAITED HER GUESTS SHE SURVEYED THE TABLE WITH BOTH S...

6. ID: 4077-13751-0018
   Duration: 26.115 seconds
   Path: 4

In [18]:
import numpy as np

def pad_audio_to_duration(audio_bytes, target_duration=30.0):
    """
    Pad audio with silence to reach target duration.
    
    Args:
        audio_bytes: Raw audio bytes (FLAC, WAV, etc.)
        target_duration: Target duration in seconds (default: 30.0)
    
    Returns:
        Padded audio bytes in FLAC format
    """
    # Read the audio
    audio_io = io.BytesIO(audio_bytes)
    audio_data, sample_rate = sf.read(audio_io)
    
    # Calculate current duration
    current_duration = len(audio_data) / sample_rate
    
    # If already at or above target, return original
    if current_duration >= target_duration:
        return audio_bytes
    
    # Calculate number of samples needed for target duration
    target_samples = int(target_duration * sample_rate)
    current_samples = len(audio_data)
    padding_samples = target_samples - current_samples
    
    # Create silence padding (zeros)
    if audio_data.ndim == 1:  # Mono
        silence = np.zeros(padding_samples, dtype=audio_data.dtype)
    else:  # Stereo or multi-channel
        silence = np.zeros((padding_samples, audio_data.shape[1]), dtype=audio_data.dtype)
    
    # Append silence to the end
    padded_audio = np.concatenate([audio_data, silence])
    
    # Write to bytes buffer
    output_buffer = io.BytesIO()
    sf.write(output_buffer, padded_audio, sample_rate, format='FLAC')
    output_buffer.seek(0)
    
    return output_buffer.read()


# Example: Pad filtered audio files to 30 seconds
print("=" * 70)
print("Padding filtered audio files to 30 seconds")
print("=" * 70)
print()

padded_audio_data = []

for idx, item in enumerate(filtered_items[:5], 1):  # Show first 5 as examples
    # Get original audio
    original_item = [x for x in dataset if x['id'] == item['id']][0]
    original_bytes = original_item['audio']['bytes']
    
    # Pad to 30s
    padded_bytes = pad_audio_to_duration(original_bytes, target_duration=30.0)
    
    # Verify new duration
    padded_io = io.BytesIO(padded_bytes)
    padded_info = sf.info(padded_io)
    
    print(f"{idx}. ID: {item['id']}")
    print(f"   Original duration: {item['duration']:.3f}s")
    print(f"   Padded duration:   {padded_info.duration:.3f}s")
    print(f"   Added silence:     {padded_info.duration - item['duration']:.3f}s")
    print()
    
    padded_audio_data.append({
        'id': item['id'],
        'audio_bytes': padded_bytes,
        'original_duration': item['duration'],
        'padded_duration': padded_info.duration,
        'text': item['text']
    })

print("=" * 70)
print(f"Successfully padded {len(padded_audio_data)} audio files to 30 seconds")
print("=" * 70)


Padding filtered audio files to 30 seconds

1. ID: 8224-274381-0002
   Original duration: 24.540s
   Padded duration:   30.000s
   Added silence:     5.460s

2. ID: 3570-5696-0003
   Original duration: 25.115s
   Padded duration:   30.000s
   Added silence:     4.885s

3. ID: 2961-960-0000
   Original duration: 27.180s
   Padded duration:   30.000s
   Added silence:     2.820s

4. ID: 5639-40744-0031
   Original duration: 28.420s
   Padded duration:   30.000s
   Added silence:     1.580s

5. ID: 1995-1836-0004
   Original duration: 33.910s
   Padded duration:   33.910s
   Added silence:     0.000s

Successfully padded 5 audio files to 30 seconds


In [19]:
# Step 1: Group audio clips by speaker and chapter to reconstruct longer segments
from collections import defaultdict

print("=" * 70)
print("Reconstructing longer audio segments from LibriSpeech")
print("=" * 70)
print()

# Group clips by speaker_id and chapter_id
grouped_clips = defaultdict(list)

for item in dataset:
    key = (item['speaker_id'], item['chapter_id'])
    audio_bytes = item['audio']['bytes']
    duration = calculate_audio_duration(audio_bytes)
    
    grouped_clips[key].append({
        'id': item['id'],
        'audio_bytes': audio_bytes,
        'duration': duration,
        'text': item['text'],
        'audio': item['audio']
    })

# Sort clips within each group by ID to maintain order
for key in grouped_clips:
    grouped_clips[key].sort(key=lambda x: x['id'])

# Calculate total duration per group
group_durations = {}
for key, clips in grouped_clips.items():
    total_duration = sum(clip['duration'] for clip in clips)
    group_durations[key] = {
        'total_duration': total_duration,
        'num_clips': len(clips),
        'clips': clips
    }

# Sort groups by total duration (longest first)
sorted_groups = sorted(group_durations.items(), 
                       key=lambda x: x[1]['total_duration'], 
                       reverse=True)

print(f"Total groups (speaker + chapter combinations): {len(sorted_groups)}")
print()
print("Top 10 longest groups:")
print("-" * 70)
for i, (key, info) in enumerate(sorted_groups[:10], 1):
    speaker_id, chapter_id = key
    print(f"{i}. Speaker {speaker_id}, Chapter {chapter_id}")
    print(f"   Clips: {info['num_clips']}, Total duration: {info['total_duration']:.2f}s")
print()

# Find groups with at least 30s of audio
groups_over_30s = [(k, v) for k, v in sorted_groups if v['total_duration'] >= 30.0]
print(f"Groups with ≥30s of audio: {len(groups_over_30s)}")
print("=" * 70)


Reconstructing longer audio segments from LibriSpeech

Total groups (speaker + chapter combinations): 87

Top 10 longest groups:
----------------------------------------------------------------------
1. Speaker 5639, Chapter 40744
   Clips: 42, Total duration: 496.69s
2. Speaker 672, Chapter 122797
   Clips: 75, Total duration: 496.16s
3. Speaker 8230, Chapter 279154
   Clips: 44, Total duration: 494.80s
4. Speaker 1188, Chapter 133604
   Clips: 45, Total duration: 491.93s
5. Speaker 2300, Chapter 131720
   Clips: 42, Total duration: 491.19s
6. Speaker 7729, Chapter 102255
   Clips: 47, Total duration: 489.95s
7. Speaker 2094, Chapter 142345
   Clips: 61, Total duration: 485.64s
8. Speaker 3575, Chapter 170457
   Clips: 57, Total duration: 483.63s
9. Speaker 4507, Chapter 16021
   Clips: 60, Total duration: 483.30s
10. Speaker 8455, Chapter 210777
   Clips: 71, Total duration: 481.85s

Groups with ≥30s of audio: 85


In [22]:
# Step 2: Concatenate clips and extract exactly 30s segments
print()
print("=" * 70)
print("Extracting exactly 30s segments")
print("=" * 70)
print()

def concatenate_audio_clips(clips):
    """Concatenate multiple audio clips into one continuous audio."""
    all_audio_data = []
    sample_rate = None
    
    for clip in clips:
        audio_io = io.BytesIO(clip['audio_bytes'])
        audio_data, sr = sf.read(audio_io)
        
        if sample_rate is None:
            sample_rate = sr
        elif sample_rate != sr:
            raise ValueError(f"Sample rate mismatch: {sample_rate} vs {sr}")
        
        all_audio_data.append(audio_data)
    
    # Concatenate all audio
    concatenated = np.concatenate(all_audio_data)
    return concatenated, sample_rate


def extract_30s_segment(audio_data, sample_rate, start_sample=0):
    """Extract exactly 30s from audio data."""
    samples_per_30s = int(30.0 * sample_rate)
    
    if start_sample + samples_per_30s > len(audio_data):
        return None
    
    segment = audio_data[start_sample:start_sample + samples_per_30s]
    
    # Convert to bytes
    output_buffer = io.BytesIO()
    sf.write(output_buffer, segment, sample_rate, format='FLAC')
    output_buffer.seek(0)
    
    return output_buffer.read()


# Extract 320 samples of exactly 30s
thirty_second_samples = []
target_samples = 320

for (speaker_id, chapter_id), info in groups_over_30s:
    if len(thirty_second_samples) >= target_samples:
        break
    
    clips = info['clips']
    
    # Concatenate all clips in this group
    try:
        concatenated_audio, sample_rate = concatenate_audio_clips(clips)
        concatenated_duration = len(concatenated_audio) / sample_rate
        
        # Extract 30s segments (can extract multiple from one group)
        num_possible_segments = int(concatenated_duration / 30.0)
        
        for seg_idx in range(num_possible_segments):
            if len(thirty_second_samples) >= target_samples:
                break
            
            start_sample = seg_idx * int(30.0 * sample_rate)
            segment_bytes = extract_30s_segment(concatenated_audio, sample_rate, start_sample)
            
            if segment_bytes:
                # Verify duration
                verify_io = io.BytesIO(segment_bytes)
                verify_info = sf.info(verify_io)
                
                # Collect all text from clips in this segment
                texts = [clip['text'] for clip in clips]
                combined_text = ' '.join(texts)
                
                thirty_second_samples.append({
                    'sample_id': len(thirty_second_samples),
                    'speaker_id': speaker_id,
                    'chapter_id': chapter_id,
                    'segment_index': seg_idx,
                    'audio_bytes': segment_bytes,
                    'duration': verify_info.duration,
                    'sample_rate': verify_info.samplerate,
                    'text': combined_text,
                    'source_ids': [clip['id'] for clip in clips]
                })
                
                print(f"Sample {len(thirty_second_samples)}: Speaker {speaker_id}, "
                      f"Chapter {chapter_id}, Segment {seg_idx}, "
                      f"Duration: {verify_info.duration:.3f}s")
    
    except Exception as e:
        print(f"Error processing group ({speaker_id}, {chapter_id}): {e}")
        continue

print()
print("=" * 70)
print(f"Successfully created {len(thirty_second_samples)} samples of exactly 30s")
print("=" * 70)



Extracting exactly 30s segments

Sample 1: Speaker 5639, Chapter 40744, Segment 0, Duration: 30.000s
Sample 2: Speaker 5639, Chapter 40744, Segment 1, Duration: 30.000s
Sample 3: Speaker 5639, Chapter 40744, Segment 2, Duration: 30.000s
Sample 4: Speaker 5639, Chapter 40744, Segment 3, Duration: 30.000s
Sample 5: Speaker 5639, Chapter 40744, Segment 4, Duration: 30.000s
Sample 6: Speaker 5639, Chapter 40744, Segment 5, Duration: 30.000s
Sample 7: Speaker 5639, Chapter 40744, Segment 6, Duration: 30.000s
Sample 8: Speaker 5639, Chapter 40744, Segment 7, Duration: 30.000s
Sample 9: Speaker 5639, Chapter 40744, Segment 8, Duration: 30.000s
Sample 10: Speaker 5639, Chapter 40744, Segment 9, Duration: 30.000s
Sample 11: Speaker 5639, Chapter 40744, Segment 10, Duration: 30.000s
Sample 12: Speaker 5639, Chapter 40744, Segment 11, Duration: 30.000s
Sample 13: Speaker 5639, Chapter 40744, Segment 12, Duration: 30.000s
Sample 14: Speaker 5639, Chapter 40744, Segment 13, Duration: 30.000s
Sampl

In [23]:
# Step 3: Save the 30s samples to disk
import os
import json
from pathlib import Path

# Create output directory
output_dir = Path("/home/ubuntu/bench_serving_v2/librispeech_30s_samples")
audio_dir = output_dir / "audio"
output_dir.mkdir(exist_ok=True)
audio_dir.mkdir(exist_ok=True)

print()
print("=" * 70)
print("Saving 30s samples to disk")
print("=" * 70)
print()

# Prepare metadata
metadata = {
    'dataset': 'LibriSpeech',
    'num_samples': len(thirty_second_samples),
    'target_duration': 30.0,
    'samples': []
}

# Save each audio file and collect metadata
for sample in thirty_second_samples:
    # Create filename
    filename = f"sample_{sample['sample_id']:03d}_spk{sample['speaker_id']}_ch{sample['chapter_id']}_seg{sample['segment_index']}.flac"
    audio_path = audio_dir / filename
    
    # Save audio file
    with open(audio_path, 'wb') as f:
        f.write(sample['audio_bytes'])
    
    # Add to metadata (without audio_bytes to keep it small)
    metadata['samples'].append({
        'sample_id': sample['sample_id'],
        'filename': filename,
        'speaker_id': sample['speaker_id'],
        'chapter_id': sample['chapter_id'],
        'segment_index': sample['segment_index'],
        'duration': sample['duration'],
        'sample_rate': sample['sample_rate'],
        'text': sample['text'],
        'source_ids': sample['source_ids']
    })
    
    print(f"Saved: {filename} ({sample['duration']:.3f}s)")

# Save metadata JSON
metadata_path = output_dir / "metadata.json"
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print()
print("=" * 70)
print(f"✓ Saved {len(thirty_second_samples)} audio files to: {audio_dir}")
print(f"✓ Saved metadata to: {metadata_path}")
print("=" * 70)
print()
print("Summary:")
print(f"  - Audio files: {audio_dir}")
print(f"  - Metadata: {metadata_path}")
print(f"  - Total samples: {len(thirty_second_samples)}")
print(f"  - Duration per sample: 30.000s")
print(f"  - Total audio duration: {len(thirty_second_samples) * 30.0:.1f}s ({len(thirty_second_samples) * 30.0 / 60:.1f} minutes)")
print("=" * 70)



Saving 30s samples to disk

Saved: sample_000_spk5639_ch40744_seg0.flac (30.000s)
Saved: sample_001_spk5639_ch40744_seg1.flac (30.000s)
Saved: sample_002_spk5639_ch40744_seg2.flac (30.000s)
Saved: sample_003_spk5639_ch40744_seg3.flac (30.000s)
Saved: sample_004_spk5639_ch40744_seg4.flac (30.000s)
Saved: sample_005_spk5639_ch40744_seg5.flac (30.000s)
Saved: sample_006_spk5639_ch40744_seg6.flac (30.000s)
Saved: sample_007_spk5639_ch40744_seg7.flac (30.000s)
Saved: sample_008_spk5639_ch40744_seg8.flac (30.000s)
Saved: sample_009_spk5639_ch40744_seg9.flac (30.000s)
Saved: sample_010_spk5639_ch40744_seg10.flac (30.000s)
Saved: sample_011_spk5639_ch40744_seg11.flac (30.000s)
Saved: sample_012_spk5639_ch40744_seg12.flac (30.000s)
Saved: sample_013_spk5639_ch40744_seg13.flac (30.000s)
Saved: sample_014_spk5639_ch40744_seg14.flac (30.000s)
Saved: sample_015_spk5639_ch40744_seg15.flac (30.000s)
Saved: sample_016_spk672_ch122797_seg0.flac (30.000s)
Saved: sample_017_spk672_ch122797_seg1.flac (30

In [ ]:
# Step 4: How to load the saved data later
print()
print("=" * 70)
print("Example: Loading saved 30s samples")
print("=" * 70)
print()

# Load metadata
with open(metadata_path, 'r') as f:
    loaded_metadata = json.load(f)

print(f"Dataset: {loaded_metadata['dataset']}")
print(f"Number of samples: {loaded_metadata['num_samples']}")
print(f"Target duration: {loaded_metadata['target_duration']}s")
print()

# Example: Load first 3 audio files
print("First 3 samples:")
print("-" * 70)
for i, sample_meta in enumerate(loaded_metadata['samples'][:3], 1):
    audio_path = audio_dir / sample_meta['filename']
    
    # Read audio file
    with open(audio_path, 'rb') as f:
        audio_bytes = f.read()
    
    # Verify duration
    audio_io = io.BytesIO(audio_bytes)
    info = sf.info(audio_io)
    
    print(f"{i}. {sample_meta['filename']}")
    print(f"   Duration: {info.duration:.3f}s")
    print(f"   Speaker: {sample_meta['speaker_id']}, Chapter: {sample_meta['chapter_id']}")
    print(f"   Text preview: {sample_meta['text'][:80]}...")
    print()

print("=" * 70)
print("Ready to use for benchmarking!")
print("=" * 70)


In [24]:
from IPython.display import Audio, display

# Listen to one of the 30s samples
# Choose which sample to listen to (0 to len(thirty_second_samples)-1)
sample_index = 0

if thirty_second_samples:
    sample = thirty_second_samples[sample_index]
    
    print("=" * 70)
    print(f"Playing Sample {sample['sample_id']}")
    print("=" * 70)
    print(f"Speaker ID: {sample['speaker_id']}")
    print(f"Chapter ID: {sample['chapter_id']}")
    print(f"Duration: {sample['duration']:.3f}s")
    print(f"Sample Rate: {sample['sample_rate']} Hz")
    print()
    print("Transcription:")
    print(sample['text'])
    print("=" * 70)
    print()
    
    # Display audio player
    display(Audio(data=sample['audio_bytes'], rate=sample['sample_rate']))
else:
    print("No samples available yet. Run the previous cells first!")


Playing Sample 0
Speaker ID: 5639
Chapter ID: 40744
Duration: 30.000s
Sample Rate: 16000 Hz

Transcription:
ELEVEN O'CLOCK HAD STRUCK IT WAS A FINE CLEAR NIGHT THEY WERE THE ONLY PERSONS ON THE ROAD AND THEY SAUNTERED LEISURELY ALONG TO AVOID PAYING THE PRICE OF FATIGUE FOR THE RECREATION PROVIDED FOR THE TOLEDANS IN THEIR VALLEY OR ON THE BANKS OF THEIR RIVER SECURE AS HE THOUGHT IN THE CAREFUL ADMINISTRATION OF JUSTICE IN THAT CITY AND THE CHARACTER OF ITS WELL DISPOSED INHABITANTS THE GOOD HIDALGO WAS FAR FROM THINKING THAT ANY DISASTER COULD BEFAL HIS FAMILY RODOLFO AND HIS COMPANIONS WITH THEIR FACES MUFFLED IN THEIR CLOAKS STARED RUDELY AND INSOLENTLY AT THE MOTHER THE DAUGHTER AND THE SERVANT MAID IN A MOMENT HE COMMUNICATED HIS THOUGHTS TO HIS COMPANIONS AND IN THE NEXT MOMENT THEY RESOLVED TO TURN BACK AND CARRY HER OFF TO PLEASE RODOLFO FOR THE RICH WHO ARE OPEN HANDED ALWAYS FIND PARASITES READY TO ENCOURAGE THEIR BAD PROPENSITIES AND THUS TO CONCEIVE THIS WICKED DESIGN TO C

In [10]:
# Calculate durations for all items in the dataset
durations = []

for item in dataset:
    

    audio_bytes = item['audio']['bytes']
    duration = calculate_audio_duration(audio_bytes)
    durations.append(duration)

# Calculate statistics
total_duration = sum(durations)
avg_duration = total_duration / len(durations)
min_duration = min(durations)
max_duration = max(durations)


print("=" * 60)
print("Audio Duration Statistics")
print("=" * 60)
print(f"Total samples: {len(dataset)}")
print(f"Total duration: {total_duration:.2f} seconds ({total_duration/60:.2f} minutes)")
print(f"Average duration: {avg_duration:.2f} seconds")
print(f"Min duration: {min_duration:.2f} seconds")
print(f"Max duration: {max_duration:.2f} seconds")
print("=" * 60)


Audio Duration Statistics
Total samples: 2620
Total duration: 19452.48 seconds (324.21 minutes)
Average duration: 7.42 seconds
Min duration: 1.28 seconds
Max duration: 34.95 seconds
